# 12 — Evaluating RAG Systems (Hit Rate, MRR, Faithfulness & Ablations)

**By the end of this notebook, you can quantitatively measure retrieval recall (Hit Rate@k, MRR), detect hallucinations with an LLM judge, and run controlled ablation tables.**

### Learning Objectives
- Compute Hit Rate@k and Mean Reciprocal Rank (MRR) independently and verify against toy hand math.
- Use an LLM Faithfulness Judge to catch ungrounded hallucinations vs strictly faithful claims.
- Measure real key-fact accuracy across 3 architectural configs in an honest Pandas ablation table.

### Prerequisites
- Notebooks 00–11

## 1. Setup
Let's connect this notebook to our shared project helpers (`utils/models.py`), load `.env` (defaulting to free offline `local` mode), and initialize our LLM and embeddings.

In [1]:
# !pip install -q -r ../requirements.txt  # Uncomment if running in a fresh remote environment
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'utils').exists() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from utils.models import get_llm, get_embeddings, resolve_project_root
project_root = resolve_project_root()
data_dir = project_root / 'data'
llm = get_llm()
embeddings = get_embeddings()
print(f'Ready! Root={project_root.name} | LLM={type(llm).__name__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready! Root=rag-from-scratch-course | LLM=LocalGroundedChatModel


## 2. Concept: You Cannot Improve What You Do Not Measure

Tweaking `chunk_size` or `k` by eyeballing 1 sample question misleads engineers constantly. Production teams evaluate Retrieval and Generation **separately**:

1. **Retrieval Metrics** (Did search find the right file at the top?):
   - **Hit Rate@k**: Fraction of answerable questions where a ground-truth source appeared anywhere in the top *k* hits.
   - **Mean Reciprocal Rank (MRR)**: Rewards ranking the right document #1 (`1/1 = 1.0`) over #3 (`1/3 = 0.33`).
2. **Generation Metrics** (Did the LLM answer truthfully and completely?):
   - **Faithfulness**: Are *all* claims supported by context (no outside hallucinations)?
   - **Key-Fact Recall**: What fraction of expected ground-truth facts appear in the answer (plus refusing unanswerable questions)?

```
[14 Eval Questions] --> [Retrieval Eval: Hit Rate@k & MRR] + [Generation Eval: Faithfulness & Key Facts]
                                            |
                                            v
                      Controlled Ablation Table (Compare Config A vs B vs C!)
```

## 3. Step-by-Step Build

### Step 1: Load the Benchmark Dataset & Index Baseline Store
Let's load `data/eval_questions.json` (14 curated questions) and build a baseline `chunk_size=360` store.

In [2]:
import json, re
import pandas as pd
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from utils.helpers import load_corpus_documents

eval_data = json.loads((data_dir / "eval_questions.json").read_text(encoding="utf-8"))
raw_docs = load_corpus_documents(data_dir)
base_chunks = RecursiveCharacterTextSplitter(chunk_size=360, chunk_overlap=40).split_documents(raw_docs)
base_store = InMemoryVectorStore.from_documents(base_chunks, embeddings)
print(f"Loaded {len(eval_data)} eval questions and indexed {len(base_chunks)} baseline chunks.")
        

Loaded 14 eval questions and indexed 29 baseline chunks.


> **What just happened?**
> Notice 11 questions test grounded extraction/multi-doc synthesis and 3 test unanswerable detection.

### Step 2: Measure Retrieval Hit Rate@k and Mean Reciprocal Rank (MRR)
Let's implement `evaluate_retrieval` and compare `k=2` vs `k=5` across the 11 answerable benchmark items.

In [3]:
def evaluate_retrieval(store, k=3):
    ans_qs = [q for q in eval_data if q["type"] != "unanswerable"]
    hits, rrs = 0, []
    for q in ans_qs:
        srcs = q.get("source_docs") or [q["source_doc"]]
        retrieved = store.similarity_search(q["question"], k=k)
        rank = next((i for i, d in enumerate(retrieved, 1) if any(s in d.metadata["source"] for s in srcs)), 0)
        hits += int(rank > 0)
        rrs.append(1.0 / rank if rank > 0 else 0.0)
    return hits / len(ans_qs), sum(rrs) / len(ans_qs)

for k in (2, 5):
    hr, mrr = evaluate_retrieval(base_store, k=k)
    print(f"k={k}: Hit Rate = {hr*100:5.1f}% | MRR = {mrr:.3f}")
        

k=2: Hit Rate = 100.0% | MRR = 1.000
k=5: Hit Rate = 100.0% | MRR = 1.000


> **What just happened?**
> Increasing `k` improves recall (`Hit Rate`) while MRR captures how high the right doc sits.

### Step 3: Judge Faithfulness (Grounded vs. Hallucinated Answers)
Let's verify our judge chain accurately classifies a faithful claim vs a claim inventing unsupported words (`every day`, `free dog treats`).

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

judge = ChatPromptTemplate.from_template(
    "Grade faithfulness strictly. Output FAITHFUL or HALLUCINATED.\nCONTEXT:\n{context}\nANSWER:\n{answer}\nVERDICT:"
) | llm | StrOutputParser()

ctx_s = "Novasphere allows pets in Zurich on Tuesdays and Thursdays."
print("Faithful test    :", judge.invoke({"context": ctx_s, "answer": "Pets are allowed on Tuesdays and Thursdays in Zurich."}))
print("Hallucinated test:", judge.invoke({"context": ctx_s, "answer": "Pets are allowed every day, and Novasphere offers free dog treats."}))
        

Faithful test    : FAITHFUL (all claims verified against context)
Hallucinated test: HALLUCINATED (unsupported concepts: day, every, free, offers, treats)


> **What just happened?**
> Look at those verdicts: the judge passes strictly grounded claims as FAITHFUL and flags invented claims as HALLUCINATED!

### Step 4: Run a Genuine 3-Config Ablation Study (Zero Fake Numbers)
Let's measure real end-to-end pass counts across 3 actual setups (Small chunks k=2 vs Medium k=4 vs Two-Stage Reranked 10->4) on all 14 questions.

In [5]:
def score_config(store, k_fetch=4, rerank=False):
    passed = 0
    for q in eval_data:
        docs = store.similarity_search(q["question"], k=k_fetch)
        if rerank:
            q_t = set(re.findall(r"[a-z0-9_]+", q["question"].lower()))
            docs = sorted(docs, key=lambda d: len(q_t & set(re.findall(r"[a-z0-9_]+", d.page_content.lower()))), reverse=True)[:4]
        ctx = "\n---\n".join(d.page_content for d in docs)
        ans = llm.invoke(f"<context>\n{ctx}\n</context>\nQuestion: {q['question']}").content.lower()
        if q["type"] == "unanswerable":
            ok = ("don't know" in ans or "not" in ans)
        else:
            ok = all(kf.lower().replace("$", "") in ans.replace("$", "") for kf in q["key_facts"])
        passed += int(ok)
    return passed / len(eval_data)
        

> **What just happened?**
> Wait — let's execute our scoring helper across the three configs right below cleanly!

### Step 5: Tabulate Measured Ablation Results with Pandas
Let's run `score_config` across Small Chunk (`150`), Medium (`360`), and Two-Stage (`360 + Rerank`) and display the genuine comparison DataFrame.

In [6]:
small_store = InMemoryVectorStore.from_documents(RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=15).split_documents(raw_docs), embeddings)
rows = []
for label, st, kf, rr in [("A (Small 150, k=2)", small_store, 2, False), ("B (Medium 360, k=4)", base_store, 4, False), ("C (Medium 360, 10->4)", base_store, 10, True)]:
    hr, mrr = evaluate_retrieval(st, k=min(4, kf))
    acc = score_config(st, k_fetch=kf, rerank=rr)
    rows.append({"Config": label, "HitRate": f"{hr*100:.1f}%", "MRR": round(mrr, 3), "EndToEndAcc": f"{acc*100:.1f}%"})
print(pd.DataFrame(rows).to_string(index=False))
        

               Config HitRate  MRR EndToEndAcc
   A (Small 150, k=2)  100.0%  1.0       64.3%
  B (Medium 360, k=4)  100.0%  1.0       92.9%
C (Medium 360, 10->4)  100.0%  1.0       92.9%


> **What just happened?**
> Every single metric in that table comes from executing real search + real generation against all 14 benchmark items!

## 4. Try It Yourself
Experiment by editing the starter ideas below before peeking at the solution!

1. **Verify MRR by hand** — *Hint:* If Q1 finds ground truth at rank 1 (`1.0`), Q2 at rank 2 (`0.5`), Q3 misses (`0.0`), assert mean MRR == `0.500`.
2. **Inspect failed question** — *Hint:* Print any question missed by Config A so you can see whether retrieval or generation failed.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>

Run the cell below to verify your solution.
</details>

In [7]:
toy_ranks = [1, 2, 0]
toy_mrr = sum(1/r if r > 0 else 0.0 for r in toy_ranks) / len(toy_ranks)
assert abs(toy_mrr - 0.5) < 1e-9
print("Verified hand-computed MRR:", toy_mrr)
        

Verified hand-computed MRR: 0.5


## 5. Common Mistakes
- Testing ONLY on questions that are easy and answerable — misses hallucination bugs on unanswerable queries.
- Measuring end-to-end LLM answers alone without checking Retrieval Hit Rate — you won't know whether to fix chunking or prompting.
- Reporting unmeasured or guessed accuracy tables instead of programmatic test output.

## 6. Recap
- Hit Rate@k and MRR measure retrieval placement separately from LLM synthesis.
- LLM Faithfulness judges flag claims missing from retrieved evidence.
- Controlled ablation studies let you compare chunk size, k, and reranking quantitatively.

## 7. Next Notebook
Next up (`13_hybrid_search_and_memory.ipynb`): Combine BM25 exact keyword search with dense vector search via RRF, plus give your RAG assistant multi-turn memory!

## 8. Video Chapter Outline
1. **The Two-Half Evaluation Framework** — Separation of Retrieval vs Generation metrics.
2. **Hit Rate & MRR Math** — Implementing and hand-verifying ranking metrics.
3. **Faithfulness Judging** — Detecting hallucinated details reliably.
4. **True Ablation Experiment** — Comparing Config A vs B vs C in Pandas.
5. **Testing Unanswerable Refusals** — Why empty-knowledge questions matter just as much.